# Compiling C code ARMv7 reference

In [15]:
%%bash
gcc -shared -fPIC -O3 -o ./armv7/libasconxof.so ./armv7/*.c

# Importing Libraries

In [16]:
import pynq
from pynq import Overlay
from pynq import allocate
import numpy as np
import ctypes
import time
import pandas as pd
import os

# Parameters

In [17]:
CRYPTO_BYTES = 64
MAX_MSG_LEN = 4096

AP_CTRL = 0x00
OUT_R_DATA = 0x18
IN_R_DATA = 0x24
LEN_DATA = 0x30

FREQ_FPGA = 100_000_000  
FREQ_CPU  = 650_000_000

# Loading Overlay

In [18]:
overlay = Overlay("./bitstream_files/xof128.bit")
ip_ascon = overlay.crypto_hash_0

lib_path = os.path.abspath("./armv7/libasconxof.so")
libascon = ctypes.CDLL(lib_path)

libascon.crypto_hash.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_ulonglong]
libascon.crypto_hash.restype = ctypes.c_int

out_hw_buffer = allocate(shape=(CRYPTO_BYTES,), dtype=np.uint8)
in_hw_buffer = allocate(shape=(MAX_MSG_LEN,), dtype=np.uint8)

out_sw_buffer = np.zeros(CRYPTO_BYTES, dtype=np.uint8)
in_sw_buffer = np.zeros(MAX_MSG_LEN, dtype=np.uint8)

# Auxiliary functions

In [19]:
def run_ascon_hw(msg_len):
    ip_ascon.write(IN_R_DATA, in_hw_buffer.physical_address)
    ip_ascon.write(OUT_R_DATA, out_hw_buffer.physical_address)
    ip_ascon.write(LEN_DATA, msg_len)
    
    ip_ascon.write(AP_CTRL, 0x01)
    
    while not (ip_ascon.read(AP_CTRL) & 0x02):
        pass
    
    out_hw_buffer.invalidate()

In [20]:
def run_ascon_sw(msg_len):
    libascon.crypto_hash(
        out_sw_buffer.ctypes.data_as(ctypes.c_void_p),
        in_sw_buffer.ctypes.data_as(ctypes.c_void_p),
        ctypes.c_ulonglong(msg_len)
    )

# Validation Tests

In [21]:
test_lengths = [0, 8, 32, 64, 128, 512, 1024, 4096]
validation_passed = True

for length in test_lengths:
    random_data = np.random.randint(0, 256, size=length, dtype=np.uint8)
    
    in_sw_buffer[:length] = random_data
    in_hw_buffer[:length] = random_data
    in_hw_buffer.flush() 
    
    run_ascon_sw(length)
    run_ascon_hw(length)
    
    is_valid = np.array_equal(out_sw_buffer, out_hw_buffer)
    if np.array_equal(out_sw_buffer, out_hw_buffer):
        print(f"{length} bytes - Hardware Correctness: Pass")
    else:
        print(f"{length} bytes - Hardware Correctness: Failed")

0 bytes - Hardware Correctness: Pass
8 bytes - Hardware Correctness: Pass
32 bytes - Hardware Correctness: Pass
64 bytes - Hardware Correctness: Pass
128 bytes - Hardware Correctness: Pass
512 bytes - Hardware Correctness: Pass
1024 bytes - Hardware Correctness: Pass
4096 bytes - Hardware Correctness: Pass


# Benchmark

In [26]:
iterations = 1000
benchmark_lengths = [64, 256, 1024, 2048, 4096]
results = []

for length in benchmark_lengths:
    start_time_sw = time.perf_counter()
    for _ in range(iterations):
        run_ascon_sw(length)
    end_time_sw = time.perf_counter()
    
    avg_time_sw = (end_time_sw - start_time_sw) / iterations
    cycles_sw = avg_time_sw * FREQ_CPU
    
    start_time_hw = time.perf_counter()
    for _ in range(iterations):
        run_ascon_hw(length)
    end_time_hw = time.perf_counter()
    
    avg_time_hw = (end_time_hw - start_time_hw) / iterations
    cycles_hw = avg_time_hw * FREQ_FPGA

    speedup_time = avg_time_sw / avg_time_hw if avg_time_hw > 0 else 0

    results.append({
        "Msg Size (Bytes)": length,
        "SW Time (us)": round(avg_time_sw * 1e6, 2),
        "SW Cycles": int(cycles_sw),
        "HW Time (us)": round(avg_time_hw * 1e6, 2),
        "HW Cycles": int(cycles_hw),
        "Speedup (Time)": round(speedup_time, 2)
    })

# Displaying benchmark results

In [27]:
df_results = pd.DataFrame(results)

display(df_results)

out_hw_buffer.close()
in_hw_buffer.close()

,Msg Size (Bytes),SW Time (us),SW Cycles,HW Time (us),HW Cycles,Speedup (Time)
0,64,188.53,122542,276.97,27696,0.68
1,256,213.03,138468,276.11,27611,0.77
2,1024,319.62,207755,276.29,27628,1.16
3,2048,459.94,298958,298.05,29805,1.54
4,4096,738.42,479970,316.62,31661,2.33
